# 7.1 — Pipeline and Persistence

**Chapter 7, section 7.2.2 and section 7.2.3.** Also the starting point for **Exercise 2** and
**Exercise 8**.

**The question this notebook answers:** what is a fitted pipeline, as an object?

The task is to predict the hourly count of bicycle rentals from calendar and weather columns.
That is the excuse. The subject is the `PipelineModel` that comes out of `fit`: **one object that
carries the fitted featurization and the fitted model together**, so that scoring unseen data is
a single call and so that the whole thing survives a round trip through disk.

Three stages, of which one is a transformer and two are estimators. The classification follows
from the interface alone, and the notebook checks it in code rather than asserting it.

Source: `Notebooks old/Spark-Example-25b-XGBoost in Spakr.ipynb`, rebuilt. The original's
estimator was `SparkXGBRegressor` from a third-party package; this notebook uses `GBTRegressor`,
exactly as the chapter's listing does, so that nothing outside Spark is required. The last
section says why that substitution is possible at all — it is the chapter's point about the
Pipeline API, not a convenience.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-7.1")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


---

## 1. The data, and the four columns that leave before the pipeline starts

Hourly bicycle rentals from the Capital Bikeshare system over 2011 and 2012. The target is
`cnt`, the number of bikes rented in that hour.

Four columns are dropped **before** the pipeline is built, and the reason is worth stating
because it is a rule about pipelines rather than about this dataset. `instant` is a row number
and `dteday` is a date whose information the calendar columns already carry. `casual` and
`registered` are the two halves of the target: `casual + registered = cnt`, exactly. A model
given them does not predict anything, it adds.

Dataset-specific preparation like this belongs **outside** the pipeline. The pipeline holds the
steps that must be reapplied identically to new data; dropping a leaking column is a decision
about this table, made once.

In [2]:
df = spark.read.csv(os.path.join(DATA, "bike-sharing.csv"), header=True, inferSchema=True)
print("rows:", df.count(), " columns:", len(df.columns))
df.select("dteday", "hr", "temp", "hum", "casual", "registered", "cnt").show(5)

rows: 17379  columns: 17
+----------+---+----+----+------+----------+---+
|    dteday| hr|temp| hum|casual|registered|cnt|
+----------+---+----+----+------+----------+---+
|2011-01-01|  0|0.24|0.81|     3|        13| 16|
|2011-01-01|  1|0.22| 0.8|     8|        32| 40|
|2011-01-01|  2|0.22| 0.8|     5|        27| 32|
|2011-01-01|  3|0.24|0.75|     3|        10| 13|
|2011-01-01|  4|0.24|0.75|     0|         1|  1|
+----------+---+----+----+------+----------+---+
only showing top 5 rows


In [3]:
# The leak, checked rather than asserted: casual + registered is cnt, in every row.
import pyspark.sql.functions as F

bad = df.filter(F.col("casual") + F.col("registered") != F.col("cnt")).count()
print("rows where casual + registered != cnt:", bad)

rows where casual + registered != cnt: 0


In [4]:
df = df.drop("instant").drop("dteday").drop("casual").drop("registered")
print(df.columns)

['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'cnt']


---

## 2. The split

Seeded, so the two counts below are the same on every machine. The training set is cached
because the pipeline that follows contains two estimator stages, and **each estimator stage is
one more pass over the training data**.

The test set is cached too, and for a different reason. A `randomSplit` is a filter over a
plan, not a materialized table, so it is re-evaluated on every action taken against either
side. Caching both sides materializes the split once and guarantees that the comparisons in
sections 5 and 6, which each score the test set twice, are comparing the same rows.

In [5]:
train, test = df.randomSplit([0.7, 0.3], seed=0)
train.cache()
test.cache()
print("training:", train.count(), " test:", test.count())

training: 12081  test: 5298


---

## 3. Three stages

* `VectorAssembler` concatenates the predictor columns into one vector column. It learns
  nothing, so it is a **transformer**.
* `VectorIndexer` looks at that vector and marks every feature with at most `maxCategories`
  distinct values as categorical — the season, the year, the weather code. It has to count
  distinct values first, so it is an **estimator**. Note what the threshold excludes: `mnth` has
  twelve values and `hr` has twenty-four, so at `maxCategories=4` both stay continuous.
* `GBTRegressor` is an **estimator**; fitting it returns a model.

In [6]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, VectorIndexer
from pyspark.ml.regression import GBTRegressor

featuresCols = df.columns
featuresCols.remove("cnt")

assembler = VectorAssembler(inputCols=featuresCols, outputCol="rawFeatures")
indexer = VectorIndexer(inputCol="rawFeatures", outputCol="features", maxCategories=4)
gbt = GBTRegressor(featuresCol="features", labelCol="cnt", maxIter=20, seed=0)

pipeline = Pipeline(stages=[assembler, indexer, gbt])
print(featuresCols)

['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']


In [7]:
# Transformer or estimator? The interface answers it; nothing else is consulted.
from pyspark.ml import Estimator, Transformer

for stage in pipeline.getStages():
    kind = "Estimator" if isinstance(stage, Estimator) else "Transformer"
    print(f"{type(stage).__name__:<18} {kind}")

VectorAssembler    Transformer
VectorIndexer      Estimator
GBTRegressor       Estimator


---

## 4. Fitting

`fit` is where the work happens. Building the pipeline above cost nothing: transformers only
describe column operations, and those are lazy like every other DataFrame transformation.

In [8]:
model = pipeline.fit(train)
predictions = model.transform(test)

predictions.select("cnt", "prediction").show(5)

+---+------------------+
|cnt|        prediction|
+---+------------------+
| 22|28.881710310685346|
| 17| 99.89708662841345|
|  7| 29.96437575028198|
| 17|27.507984207200142|
|  9| 24.64190854277712|
+---+------------------+
only showing top 5 rows


### The same three stages, with the estimators replaced

The fitted `PipelineModel` has the same number of stages in the same order. The only difference
is that each estimator has become the model it produced — and every one of those models is
itself a transformer, which is why the whole chain can be applied with a single `transform`.

In [9]:
for before, after in zip(pipeline.getStages(), model.stages):
    print(f"{type(before).__name__:<18} ->  {type(after).__name__}")

VectorAssembler    ->  VectorAssembler
VectorIndexer      ->  VectorIndexerModel
GBTRegressor       ->  GBTRegressionModel


In [10]:
# What the middle stage learned: which features it decided were categorical, and how many
# values each of them has. This is state that lives in the fitted object, not in the code.
vi = model.stages[1]
cats = vi.categoryMaps
print("features marked categorical (index -> number of categories):")
for i in sorted(cats):
    print(f"  {featuresCols[i]:<12} {len(cats[i])}")

features marked categorical (index -> number of categories):
  season       4
  yr           2
  holiday      2
  workingday   2
  weathersit   4


---

## 5. Scoring, and why it is one call

The whole argument for the Pipeline API is in the next cell. Applying the workflow to data that
was never seen during training is `model.transform(test)` and nothing else. The assembler runs
with the same column order, the indexer applies the **same** category maps it learned from the
training data, and the regressor scores the result.

To see what the alternative costs, the cell after it reproduces the same predictions by driving
the three fitted stages by hand. It gets the same answer — which is the point. Every one of
those three lines is a chance to use a different column order, a differently fitted indexer, or
a stage in the wrong place, and none of those mistakes raises an error.

In [11]:
one_call = model.transform(test).select("cnt", "prediction")
one_call.show(3)

+---+------------------+
|cnt|        prediction|
+---+------------------+
| 22|28.881710310685346|
| 17| 99.89708662841345|
|  7| 29.96437575028198|
+---+------------------+
only showing top 3 rows


In [12]:
by_hand = model.stages[2].transform(
    model.stages[1].transform(
        model.stages[0].transform(test))).select("cnt", "prediction")

assert one_call.subtract(by_hand).count() == 0
assert by_hand.subtract(one_call).count() == 0
print("the one call and the three by hand agree, row for row")

the one call and the three by hand agree, row for row


---

## 6. Persistence: the pipeline and the pipeline model are different things

Both can be saved, and the distinction matters.

* Saving the **`Pipeline`** saves the specification — the stages, their order, their parameters.
  Reloading it gives you something you still have to `fit`.
* Saving the **`PipelineModel`** saves the specification *and every parameter learned during
  training*: the indexer's category maps, the trees of the regressor, the assembler's column
  order.

Regression coefficients alone would not be enough to score a raw row, because they assume the
features were built the right way. The saved `PipelineModel` carries the feature building too.

In [13]:
import shutil

pipe_path = os.path.join(SCRATCH, "bike_pipeline")
model_path = os.path.join(SCRATCH, "bike_pipeline_model")
shutil.rmtree(pipe_path, ignore_errors=True)
shutil.rmtree(model_path, ignore_errors=True)

pipeline.save(pipe_path)
model.save(model_path)

for path in (pipe_path, model_path):
    total = sum(os.path.getsize(os.path.join(d, f))
                for d, _, fs in os.walk(path) for f in fs)
    print(f"{os.path.basename(path):<22} {total/1024:8.1f} kB on disk")

bike_pipeline               1.7 kB on disk
bike_pipeline_model        70.9 kB on disk


The unfitted pipeline is a few kilobytes of metadata. The fitted model is much larger, and the
difference **is** the learned state.

In [14]:
from pyspark.ml import Pipeline, PipelineModel

loaded_pipeline = Pipeline.load(pipe_path)
loaded_model = PipelineModel.load(model_path)

print([type(s).__name__ for s in loaded_pipeline.getStages()])
print([type(s).__name__ for s in loaded_model.stages])

['VectorAssembler', 'VectorIndexer', 'GBTRegressor']
['VectorAssembler', 'VectorIndexerModel', 'GBTRegressionModel']


In [15]:
# Score three unseen rows with the reloaded model. Note what was NOT imported to do it:
# no GBTRegressor, no VectorIndexer, no VectorAssembler.
loaded_model.transform(test.limit(3)).select("cnt", "prediction").show()

+---+------------------+
|cnt|        prediction|
+---+------------------+
| 22|28.881710310685346|
| 17| 99.89708662841345|
|  7|29.964375750281988|
+---+------------------+



In [16]:
# The reloaded model is the same model. "Same" here means to within floating-point rounding,
# not bit for bit: the tree weights make a round trip through a serialized format and come
# back differing in the last digit or two. Both sets are sorted before comparison because
# collect() makes no promise about row order.
import math

a = sorted(r.prediction for r in model.transform(test).select("prediction").collect())
b = sorted(r.prediction for r in loaded_model.transform(test).select("prediction").collect())

worst = max(abs(x - y) for x, y in zip(a, b))
assert len(a) == len(b)
assert all(math.isclose(x, y, rel_tol=1e-12) for x, y in zip(a, b))
print(f"{len(a)} predictions, largest disagreement {worst:.3e} "
      f"({sum(1 for x, y in zip(a, b) if x != y)} of them differ at all)")

5298 predictions, largest disagreement 2.842e-13 (3083 of them differ at all)


That is the honest form of the claim, and the size of the number is the point. A saved
`PipelineModel` reloads to within about $10^{-14}$ relative, not to the bit, because the learned
parameters are written out and parsed back. It is far below anything that could change a decision,
and it is not zero — an equality test on floating-point predictions across a save would fail
roughly half the time, for a reason that has nothing to do with the model.

---

## 7. Why the estimator could be swapped at all

The repository version of this example fitted `SparkXGBRegressor`, from the XGBoost package,
which is not part of MLlib and does not appear in the chapter's catalogue of algorithms. This
notebook fits `GBTRegressor` instead, and **nothing else in the pipeline changed**: not the
assembler, not the indexer, not the fit call, not the save, not the reload.

That is not a coincidence of the two classes being similar. A pipeline stage is defined by its
interface — an estimator is anything with a `fit` that returns something with a `transform`. Any
class satisfying that participates in an MLlib pipeline exactly as though it were part of Spark.

The next cell makes the same substitution once more, with a different tree ensemble, to show
that the surrounding pipeline is genuinely indifferent to what the last stage is.

In [17]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

rmse = RegressionEvaluator(labelCol="cnt", predictionCol="prediction", metricName="rmse")

for estimator in (GBTRegressor(featuresCol="features", labelCol="cnt", maxIter=20, seed=0),
                  RandomForestRegressor(featuresCol="features", labelCol="cnt", seed=0)):
    m = Pipeline(stages=[assembler, indexer, estimator]).fit(train)
    print(f"{type(estimator).__name__:<22} rmse {rmse.evaluate(m.transform(test)):7.2f}")

GBTRegressor           rmse   61.56


RandomForestRegressor  rmse  111.41


---

## What to take away

A `Pipeline` is an estimator; fitting it gives a `PipelineModel`, which is a transformer with the
same stages in the same order and every estimator replaced by what it learned. That single
object is what makes training and test data provably identical in their feature processing, and
it is what a `save` writes to disk.

The counts to remember from this run: **17,379 hourly observations**, split at seed 0 into
**12,081 training and 5,298 test rows**; three stages, of which **one is a transformer and two
are estimators**, so fitting the pipeline makes **two passes** over the training data before the
model exists.

*Timings and RMSE values here are machine-dependent; the row counts and the stage structure are
not.*

In [18]:
spark.stop()